# 145 — LoRA Architecture Ablation

**What you'll learn:**
- The LoRA math: W' = W + B×A where trainable params = r×(d_in + d_out) per layer
- How rank `r` controls capacity vs parameter count tradeoff
- Which transformer modules to adapt: q/v only vs full attention vs attention+FFN
- Why ablation matters: avoid over-parameterizing before committing to training time
- How to read the 4 ablation configs and what each tests
- How to build a comparison table from sweep results

---
**Source:** `examples/145-lora-architecture-ablation/`  
**Model:** `HuggingFaceTB/SmolLM2-135M-Instruct`  
**Part A** is CPU-safe. **Part B** requires a GPU (recommended — CPU is slow but works for small models).

In [ ]:
%pip install -q transformers peft datasets torch

## Part A — CPU-Safe Concept Demonstrations

All cells below run without a GPU and without downloading any model weights.

### A1 — The LoRA Math

Standard fine-tuning updates W (shape d_out × d_in) = d_out × d_in parameters

LoRA freezes W and learns:
```
  ΔW = B × A
  where:
    A ∈ ℝ^(r × d_in)    r rows, d_in columns  (projects DOWN)
    B ∈ ℝ^(d_out × r)   d_out rows, r columns  (projects UP)

Trainable params per layer = r × d_in + d_out × r = r(d_in + d_out)
Full fine-tune params      = d_in × d_out

Example: q_proj with d_in = d_out = 576, r = 8:
  LoRA : 8 × 576 + 576 × 8 = 9,216 params
  Full : 576 × 576 = 331,776 params ← 36× more
```

The scaling factor `alpha` controls how much weight the LoRA update gets during the forward pass:
```
Forward pass:
  y = W·x + (alpha/r) × B·A·x

where alpha is a scaling hyperparameter (common choices: alpha = r or alpha = 2r)

Setting alpha = 2r means the LoRA update is scaled by 2.0 at initialization.
This is the default in most PEFT implementations.
```

In [ ]:
# A2 — Rank impact: how r affects trainable parameter count
# Shows the key LoRA formula: params = r * (d_in + d_out) per layer
# Uses SmolLM2-135M approximate dimensions.

def lora_params(r: int, d_in: int, d_out: int) -> int:
    """LoRA trainable params for one adapted projection: r*(d_in + d_out)"""
    return r * (d_in + d_out)

# SmolLM2-135M dimensions
D = 576       # hidden size
N_LAYERS = 30  # transformer layers

# Full fine-tuning baseline for one projection
full_ft = D * D

print(f"Model hidden size: d={D}, layers={N_LAYERS}")
print(f"Full fine-tune per projection: {full_ft:,} params\n")

print(f"{'Rank r':<8} {'Params/proj':<14} {'q+v, all layers':<18} {'vs full FT':<12} {'Generalization risk'}")
print("-" * 74)

risk = {4: "Low — underfitting possible for complex tasks",
        8: "Low-medium — good default",
        16: "Medium — standard quality",
        32: "Medium-high — close to full FT quality",
        64: "High — may overfit small datasets"}

for r in [4, 8, 16, 32, 64]:
    per_proj = lora_params(r, D, D)
    total_qv = per_proj * 2 * N_LAYERS  # q_proj + v_proj
    ratio = full_ft * 2 * N_LAYERS / total_qv
    print(f"{r:<8} {per_proj:<14,} {total_qv:<18,} {ratio:<12.1f}x  {risk[r]}")

print()
print("Key insight: r=8 gives ~36× fewer params than full fine-tuning per proj,")
print("yet often reaches within 1-3% of full fine-tuning quality on many tasks.")

### A3 — Which Modules to Adapt: target_modules Guide

Each transformer layer contains several weight matrices. `target_modules` tells LoRA which of these receive adapters.

```
Transformer Layer
┌──────────────────────────────────────────────────────────┐
│  Self-Attention Block                                    │
│    q_proj  ← Query  (common LoRA target)                │
│    k_proj  ← Key                                         │
│    v_proj  ← Value  (common LoRA target)                │
│    o_proj  ← Output projection                           │
├──────────────────────────────────────────────────────────┤
│  MLP / FFN Block                                         │
│    gate_proj  ← SwiGLU gate                              │
│    up_proj    ← SwiGLU up projection                     │
│    down_proj  ← dimension reduction                      │
└──────────────────────────────────────────────────────────┘
```

| Strategy | Modules | Params multiplier | Use case |
|----------|---------|------------------|----------|
| Minimal | `["q_proj", "v_proj"]` | 1× | Fast training, most tasks |
| Full attention | `["q_proj", "k_proj", "v_proj", "o_proj"]` | 2× | Tasks needing precise attention |
| Attention + FFN | all 7 projections | ~5× | Domain adaptation, large distribution shift |

> **Rule of thumb:** Start with `q_proj + v_proj` — it's the classic choice from the original LoRA paper. Add `k_proj` and `o_proj` if the task needs precise key-value attention (e.g., complex reasoning). Only add FFN layers if you're doing heavy domain adaptation (e.g., fine-tuning on medical/legal text).

In [ ]:
# A4 — Print the 4 ablation configs from workflow.py
# These mirror ABLATION_CONFIGS in src/workflow.py exactly.

ABLATION_CONFIGS = [
    {"r": 4,  "target_modules": ["q_proj"],                             "label": "qv_r4"},
    {"r": 8,  "target_modules": ["q_proj", "v_proj"],                   "label": "qkvo_r8"},
    {"r": 16, "target_modules": ["q_proj", "v_proj"],                   "label": "qkvo_r16"},
    {"r": 32, "target_modules": ["q_proj", "v_proj", "o_proj"],         "label": "qkvo_ffn_r16"},
]

# Model params for computing ratios
D = 576
N_LAYERS = 30
TOTAL_PARAMS = 135_000_000  # SmolLM2-135M

print("The 4 ablation configs\n")
print(f"{'Config':<15} {'r':<5} {'Modules':<35} {'LoRA params':<14} {'% of model'}")
print("-" * 80)

for cfg in ABLATION_CONFIGS:
    r = cfg["r"]
    n_mods = len(cfg["target_modules"])
    lora_params = 2 * r * D * n_mods * N_LAYERS  # r*(d_in + d_out) per proj
    pct = 100 * lora_params / TOTAL_PARAMS
    modules_str = " + ".join(cfg["target_modules"])
    print(f"{cfg['label']:<15} {r:<5} {modules_str:<35} {lora_params:<14,} {pct:.4f}%")

print()
print("What the ablation tests:")
print("  qv_r4:       Does low rank on q-only work at all?")
print("  qkvo_r8:     Standard q+v with moderate rank — baseline config")
print("  qkvo_r16:    Same modules, doubled rank — does more capacity help?")
print("  qkvo_ffn_r16: Add o_proj — does output projection matter?")

In [ ]:
# A5 — Simulate loss trajectories for each config (mock data, no training)
# Shows the pattern we'd expect to see from the ablation sweep.

import math

def mock_loss_curve(initial_loss: float, final_loss: float, steps: int, noise: float = 0.02):
    """Simulate a smooth loss curve with slight noise."""
    import random
    random.seed(42)
    curve = []
    for i in range(steps):
        t = i / (steps - 1)
        # Exponential decay from initial to final
        base = initial_loss * math.exp(-3 * t) + final_loss * (1 - math.exp(-3 * t))
        noisy = base + random.gauss(0, noise)
        curve.append(round(max(noisy, final_loss * 0.9), 3))
    return curve

CONFIGS = {
    "qv_r4":      {"init": 2.8, "final": 2.15},
    "qkvo_r8":    {"init": 2.8, "final": 1.95},
    "qkvo_r16":   {"init": 2.8, "final": 1.80},
    "qkvo_ffn_r16": {"init": 2.8, "final": 1.65},
}

STEPS = 20
PRINT_AT = [0, 5, 10, 15, 19]

print(f"Mock loss trajectories (simulated, {STEPS} steps)\n")
print(f"{'Config':<18}", end="")
for s in PRINT_AT:
    print(f"  step {s:<4}", end="")
print(f"  final")
print("-" * 70)

for name, params in CONFIGS.items():
    curve = mock_loss_curve(params["init"], params["final"], STEPS)
    print(f"{name:<18}", end="")
    for s in PRINT_AT:
        print(f"  {curve[s]:<8.3f}", end="")
    print(f"  {curve[-1]:.3f}")

print()
print("Expected pattern:")
print("  - Higher rank → lower final loss (more capacity)")
print("  - More modules → lower final loss (covers more of the model)")
print("  - Diminishing returns: r16→r32 gain < r4→r8 gain")
print("  - Adding o_proj gives a meaningful boost beyond q+v alone")

In [ ]:
# A6 — Rank sensitivity: sweep r from 4 to 64 with fixed modules q+v
# Shows how trainable params scale with rank for a fixed module set.

D = 576
N_LAYERS = 30
TOTAL_PARAMS = 135_000_000
MODULES_FIXED = ["q_proj", "v_proj"]
N_MODS = len(MODULES_FIXED)

print(f"Rank sensitivity analysis")
print(f"Fixed modules: {MODULES_FIXED}, d={D}, layers={N_LAYERS}\n")
print(f"{'Rank r':<10} {'LoRA params':<15} {'% of model':<14} {'Est. VRAM delta (fp16)'}")
print("-" * 60)

for r in [4, 8, 16, 32, 64, 128]:
    # r*(d_in + d_out) per proj = 2*r*d for square projections
    lora_params = 2 * r * D * N_MODS * N_LAYERS
    pct = 100 * lora_params / TOTAL_PARAMS
    # Adapter VRAM: 2 bytes/param (fp16) × 3 (A,B + optimizer state Adam)
    vram_mb = lora_params * 2 * 3 / 1e6
    print(f"{r:<10} {lora_params:<15,} {pct:<14.4f} ~{vram_mb:.1f} MB")

print()
print("Note: adapter VRAM is tiny compared to backbone (135M × 0.5 bytes = 67 MB for NF4).")
print("Rank can often be increased significantly with minimal VRAM cost.")
print("The constraint is usually convergence quality, not memory.")

---
## Exercise 1: Add a 5th Ablation Config

The 4 configs in `ABLATION_CONFIGS` test attention-focused adaptations.

**Add a 5th config that targets only FFN (MLP) layers:**
- Label: `"ffn_only_r8"`
- Rank: `r=8`
- Modules: `["gate_proj", "up_proj", "down_proj"]`

Then extend the parameter table to include it and answer:
- How many trainable params does FFN-only r=8 add?
- How does it compare to `qkvo_r8` (which uses attention projections)?

In [ ]:
# Exercise 1: Add a 5th ablation config targeting only FFN layers

ABLATION_CONFIGS = [
    {"r": 4,  "target_modules": ["q_proj"],                             "label": "qv_r4"},
    {"r": 8,  "target_modules": ["q_proj", "v_proj"],                   "label": "qkvo_r8"},
    {"r": 16, "target_modules": ["q_proj", "v_proj"],                   "label": "qkvo_r16"},
    {"r": 32, "target_modules": ["q_proj", "v_proj", "o_proj"],         "label": "qkvo_ffn_r16"},
    # TODO: add 5th config here — ffn_only_r8
    # {"r": ..., "target_modules": [...], "label": "ffn_only_r8"},
]

D = 576
N_LAYERS = 30
TOTAL_PARAMS = 135_000_000

# TODO: recompute the parameter table including the 5th config
print(f"{'Config':<15} {'r':<5} {'Modules':<35} {'LoRA params':<14} {'% of model'}")
print("-" * 80)
for cfg in ABLATION_CONFIGS:
    r = cfg["r"]
    n_mods = len(cfg["target_modules"])
    lora_params = 2 * r * D * n_mods * N_LAYERS
    pct = 100 * lora_params / TOTAL_PARAMS
    modules_str = " + ".join(cfg["target_modules"])
    print(f"{cfg['label']:<15} {r:<5} {modules_str:<35} {lora_params:<14,} {pct:.4f}%")

In [ ]:
# Exercise 1 — Answer Key

ABLATION_CONFIGS = [
    {"r": 4,  "target_modules": ["q_proj"],                              "label": "qv_r4"},
    {"r": 8,  "target_modules": ["q_proj", "v_proj"],                    "label": "qkvo_r8"},
    {"r": 16, "target_modules": ["q_proj", "v_proj"],                    "label": "qkvo_r16"},
    {"r": 32, "target_modules": ["q_proj", "v_proj", "o_proj"],          "label": "qkvo_ffn_r16"},
    {"r": 8,  "target_modules": ["gate_proj", "up_proj", "down_proj"],   "label": "ffn_only_r8"},
]

D = 576
N_LAYERS = 30
TOTAL_PARAMS = 135_000_000

print(f"{'Config':<15} {'r':<5} {'Modules':<40} {'LoRA params':<14} {'% of model'}")
print("-" * 82)

for cfg in ABLATION_CONFIGS:
    r = cfg["r"]
    n_mods = len(cfg["target_modules"])
    lora_params = 2 * r * D * n_mods * N_LAYERS
    pct = 100 * lora_params / TOTAL_PARAMS
    modules_str = " + ".join(cfg["target_modules"])
    print(f"{cfg['label']:<15} {r:<5} {modules_str:<40} {lora_params:<14,} {pct:.4f}%")

print()
# The FFN-only config with r=8 on gate+up+down has 3 modules vs 2 for qkvo_r8.
# So ffn_only_r8 has 50% more trainable params than qkvo_r8.
qkvo_r8_params = 2 * 8 * D * 2 * N_LAYERS
ffn_only_params = 2 * 8 * D * 3 * N_LAYERS
print(f"qkvo_r8 params    : {qkvo_r8_params:,}")
print(f"ffn_only_r8 params: {ffn_only_params:,}")
print(f"FFN-only has {ffn_only_params/qkvo_r8_params:.1f}x more params than qkvo_r8")
print()
print("When to use FFN-only: tasks requiring factual knowledge injection (e.g., medical/legal).")
print("When to use attention-only: tasks requiring reasoning or instruction following.")

---
## Exercise 2: Parameter Efficiency Ratio

**Define a metric**: `param_efficiency_ratio = quality_improvement / trainable_params`

With mock data, compute which config gives the best "bang per parameter".

Given these mock results:
| Config | Trainable params | Baseline loss | Final loss |
|--------|-----------------|---------------|------------|
| qv_r4 | 138,240 | 2.80 | 2.15 |
| qkvo_r8 | 552,960 | 2.80 | 1.95 |
| qkvo_r16 | 1,105,920 | 2.80 | 1.80 |
| qkvo_ffn_r16 | 1,658,880 | 2.80 | 1.65 |

Write a function that:
1. Takes `(baseline_loss, final_loss, trainable_params)`
2. Returns `quality_gain / (trainable_params / 1e6)` — gain per million params

In [ ]:
# Exercise 2: param-efficiency ratio = loss_reduction / (trainable_params / 1e6)

MOCK_RESULTS = [
    {"label": "qv_r4",        "trainable": 138_240,   "baseline": 2.80, "final": 2.15},
    {"label": "qkvo_r8",      "trainable": 552_960,   "baseline": 2.80, "final": 1.95},
    {"label": "qkvo_r16",     "trainable": 1_105_920, "baseline": 2.80, "final": 1.80},
    {"label": "qkvo_ffn_r16", "trainable": 1_658_880, "baseline": 2.80, "final": 1.65},
]

def param_efficiency_ratio(baseline_loss: float, final_loss: float, trainable_params: int) -> float:
    """
    Returns quality_gain per million trainable parameters.
    quality_gain = baseline_loss - final_loss (higher = better)
    """
    # TODO: implement this
    pass

# TODO: compute and print a table: config | trainable_params | loss_reduction | efficiency_ratio
print(f"{'Config':<18} {'Params':<12} {'Loss drop':<12} {'Efficiency (gain/M params)'}")
print("-" * 62)
for r in MOCK_RESULTS:
    ratio = param_efficiency_ratio(r["baseline"], r["final"], r["trainable"])
    # print(f"{r['label']:<18} {r['trainable']:<12,} {r['baseline']-r['final']:<12.2f} {ratio:.4f}")

In [ ]:
# Exercise 2 — Answer Key

MOCK_RESULTS = [
    {"label": "qv_r4",        "trainable": 138_240,   "baseline": 2.80, "final": 2.15},
    {"label": "qkvo_r8",      "trainable": 552_960,   "baseline": 2.80, "final": 1.95},
    {"label": "qkvo_r16",     "trainable": 1_105_920, "baseline": 2.80, "final": 1.80},
    {"label": "qkvo_ffn_r16", "trainable": 1_658_880, "baseline": 2.80, "final": 1.65},
]

def param_efficiency_ratio(baseline_loss: float, final_loss: float, trainable_params: int) -> float:
    """Quality gain per million trainable parameters."""
    quality_gain = baseline_loss - final_loss  # loss reduction (higher = better)
    params_millions = trainable_params / 1e6
    return round(quality_gain / params_millions, 4)

print(f"{'Config':<18} {'Params':<14} {'Loss drop':<12} {'Efficiency (gain/M params)'}")
print("-" * 65)

best_ratio = 0
best_config = ""
for r in MOCK_RESULTS:
    ratio = param_efficiency_ratio(r["baseline"], r["final"], r["trainable"])
    if ratio > best_ratio:
        best_ratio = ratio
        best_config = r["label"]
    print(f"{r['label']:<18} {r['trainable']:<14,} {r['baseline']-r['final']:<12.2f} {ratio:.4f}")

print()
print(f"Most parameter-efficient config: {best_config} (ratio = {best_ratio:.4f})")
print()
print("Insight: qv_r4 achieves the best gain per parameter — it's surprisingly efficient.")
print("This doesn't mean it's the best absolute quality, just the most parameter-efficient.")
print("Choose the metric that matches your constraint: VRAM → lower r, quality → higher r.")

---
## Part B — Full Ablation Sweep

> **Requirements:**
> - CUDA GPU recommended (CPU is usable but slow — each config takes several minutes on CPU)
> - Free GPU: [Google Colab T4 runtime](https://colab.research.google.com) → Runtime > Change runtime type > T4 GPU
> - This runs 4 training jobs sequentially — total ~5-10 minutes on T4

**What Part B demonstrates:**
- Running all 4 ablation configs on the same model
- Comparing trainable parameter counts, final loss, and training time
- Building the comparison table you'd use to choose a config for your project

In [ ]:
# B1 — GPU check (recommended but not required for small models)
import torch

if not torch.cuda.is_available():
    print("WARNING: No CUDA GPU detected.")
    print("Part B will run on CPU but will be slow (several minutes per config).")
    print("For faster results: Runtime > Change runtime type > T4 GPU")
    DEVICE = "cpu"
else:
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {vram_gb:.1f} GB")
    DEVICE = "cuda"

print(f"\nDevice: {DEVICE}")

In [ ]:
# B2 — Full ablation sweep (from workflow.py)
import sys, time
sys.path.insert(0, '/content')  # Colab path; adjust if running locally

from src.workflow import create_workflow

workflow = create_workflow()

state = {
    "model_name": "HuggingFaceTB/SmolLM2-135M-Instruct",
    "n_steps": 20,
    "results": [],
}

print(f"Running LoRA ablation on {state['model_name']}")
print(f"4 configs × {state['n_steps']} steps each")
print("This will take 5-10 minutes on T4 GPU, longer on CPU...\n")

start = time.time()
result = workflow(state)
elapsed = time.time() - start

print(f"\nCompleted in {elapsed:.0f} seconds\n")
print(f"{'Config':<15} {'Rank':<6} {'Targets':<35} {'Trainable%':<12} {'Final loss'}")
print("-" * 76)
for r in result["results"]:
    targets = " + ".join(r["targets"])
    pct = r["params"]["pct"]
    print(f"{r['label']:<15} {r['r']:<6} {targets:<35} {pct:<12.4f} {r['final_loss']}")

In [ ]:
# B3 — Build full comparison table from sweep results and analyze
# Run B2 first to populate result["results"]

print("LoRA Ablation Comparison Table\n")
print(f"{'Config':<15} {'r':<6} {'Trainable params':<20} {'%':<8} {'Final loss':<12} {'Time/step est.'}")
print("=" * 76)

for r in result["results"]:
    targets = " + ".join(r["targets"])
    params = r["params"]
    print(f"{r['label']:<15} {r['r']:<6} {params['trainable']:<20,} {params['pct']:<8.4f} {r['final_loss']:<12}")

print()

# Find best config
best_loss = min(r["final_loss"] for r in result["results"])
best = next(r for r in result["results"] if r["final_loss"] == best_loss)
min_params = min(r["params"]["trainable"] for r in result["results"])
most_efficient = next(r for r in result["results"] if r["params"]["trainable"] == min_params)

print(f"Best absolute loss    : {best['label']} (loss={best_loss})")
print(f"Fewest params         : {most_efficient['label']} ({most_efficient['params']['trainable']:,} params)")
print()
print("Observations from this sweep:")
print("  1. Higher rank generally reduces loss, but with diminishing returns")
print("  2. Adding o_proj on top of q+v gives incremental improvement")
print("  3. The 'best' config depends on your VRAM budget, not just final loss")
print("  4. For a 135M model with 20 steps, differences are small — scale up for a real task")

### LoRA Dropout — Regularization for Adapters

LoRA adapters can overfit, especially on small datasets (< 1000 examples).
LoRA dropout is applied to the adapter inputs before the A matrix:

```
Input x  →  [Dropout(p)]  →  A (down-projection)  →  B (up-projection)  →  scaled output
```

| Dataset size | Recommended dropout |
|---|---|
| < 500 examples | 0.1–0.2 |
| 500–5,000 | 0.05 |
| > 5,000 | 0.0 (no dropout) |

In PEFT: `LoraConfig(lora_dropout=0.05, ...)`

Dropout has no effect during inference since adapters are typically merged into weights.

### AdaLoRA — Adaptive Rank Distribution

Standard LoRA assigns the same rank r to every layer. AdaLoRA allocates rank **dynamically**
based on the importance of each layer's update:

```
Standard LoRA:    [r=8] [r=8] [r=8] [r=8] [r=8] [r=8] [r=8] [r=8]
AdaLoRA (budget): [r=2] [r=4] [r=8] [r=16] [r=8] [r=4] [r=2] [r=4]
                   same total parameter budget, better distributed
```

AdaLoRA identifies important layers via singular value decomposition (SVD) of the adapter
weight matrices and prunes small singular values.

**Use AdaLoRA when:** the task requires high quality and you're not sure where the capacity
should go. **Standard LoRA** is fine for most cases and trains faster.

In [ ]:
# Interaction between rank and dropout — impact on effective capacity
# Higher rank → more capacity but more overfitting risk → needs more dropout

def recommend_dropout(r: int, dataset_size: int) -> dict:
    if dataset_size < 500:
        base = 0.15
    elif dataset_size < 5000:
        base = 0.05
    else:
        base = 0.0
    # Higher rank needs slightly more regularization
    rank_adjustment = max(0, (r - 8) * 0.005)
    dropout = min(0.3, base + rank_adjustment)
    return {'r': r, 'dataset_size': dataset_size,
            'recommended_dropout': round(dropout, 3)}

print(f'{'r':>4} | {'n_samples':>10} | {'dropout':>8}')
print('-' * 30)
for r in [4, 8, 16, 32]:
    for n in [200, 2000, 10000]:
        rec = recommend_dropout(r, n)
        print(f'{r:>4} | {n:>10} | {rec["recommended_dropout"]:>8.3f}')
    print()

print('These are starting points. Always validate on a held-out eval set.')

### Merging LoRA Adapters at Inference

After training, the LoRA adapter can be **merged** back into the base model weights:

```python
# PEFT merge API
model = model.merge_and_unload()  # W_merged = W + (B @ A) * scale
```

**Advantages of merging:**
- Zero inference overhead (no extra forward passes through A and B)
- Simpler deployment (one model file instead of base + adapter)
- Compatible with quantization for serving (re-quantize the merged model)

**Keep adapter separate when:**
- You need to hot-swap multiple adapters (e.g., different languages or styles)
- You want to keep the base model frozen and serve multiple tasks
- You're using a framework like vLLM's LoRA serving API

Post-merge, the model has identical inference performance to a fully fine-tuned model.

---
## What's Next

- **LoRA paper** (Hu et al. 2021, arxiv.org/abs/2106.09685) — original paper, Section 4 has the ablations
- **AdaLoRA** (Zhang et al. 2023, arxiv.org/abs/2303.10512) — adaptive rank allocation
- **DoRA** (Liu et al. 2024, arxiv.org/abs/2402.09353) — weight-decomposed LoRA with better results
- **Example 144 — QLoRA Fine-tuning:** QLoRA uses exactly these LoRA configs with NF4 quantization
- **Example 146 — ORPO Alignment:** use ablation findings to pick the LoRA config for preference training

---
## What's Next

You've seen how different LoRA configurations trade parameter count for quality. Here's where to go deeper:

- **LoRA paper**: Hu et al. (2021) — "LoRA: Low-Rank Adaptation of Large Language Models" — https://arxiv.org/abs/2106.09685  
  The original rank sensitivity analysis is in Section 5.
- **AdaLoRA** (2023): Dynamically adjusts rank per layer based on importance scores — https://arxiv.org/abs/2303.10512
- **PEFT library target_modules guide**: https://huggingface.co/docs/peft — search for "target_modules" for model-specific lists
- **LoRA rank sensitivity study**: "The Impact of LoRA Rank on LLM Fine-tuning" (2024) — empirically tests r=1 to r=256 on LLaMA
- **Example 144**: QLoRA Fine-Tuning — combine LoRA with 4-bit quantization to run the full pipeline on a T4 GPU
- **Example 146**: ORPO Alignment — once you have a fine-tuned model, align it to preferences without a reference model

---
## Summary

| Concept | Key formula | Practical default |
|---------|-------------|-------------------|
| LoRA math | r × (d_in + d_out) per layer | r=8, q_proj + v_proj |
| Rank tradeoff | Higher r → more capacity, more params | Start r=8, double if underfitting |
| Module selection | Attention only → all attention → attention+FFN | q+v is usually enough |
| Efficiency metric | loss_reduction / trainable_params (M) | Use to compare configs |

The ablation sweep in this example gives you a systematic framework: **run small, measure, then commit to the best config for your full training run.**